# Create Gerber Foundation Awards

Builds the awards table for the **Gerber Foundation** (funder_id `4320306353`,
IAMHRF expanded list, priority `289`) from their annual national research-grant
award PDFs (2023-2025) on gerberfoundation.org. 56 grants with institution, PI
(+ degrees), USD amount, city, project title, and award year. No native grant
id (synthesized from year + institution + PI + title); maternal/infant/child
health funder.

Source parquet: `s3://openalex-ingest/awards/gerber/gerber_grants.parquet`
(produced by `scripts/local/gerber_to_s3.py`).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gerber_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gerber/gerber_grants.parquet`;

In [ ]:
%sql
-- Check row count (should be ~56)
SELECT COUNT(*) as total_projects FROM openalex.awards.gerber_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.gerber_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.gerber_raw LIMIT 5;

## Step 2: Create Gerber Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gerber_awards
USING delta
AS
WITH
gerber_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306353  -- Gerber Foundation
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'USD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        CAST(NULL AS STRING) as funder_scheme,
        'gerber' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United States' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.gerber_raw g
    CROSS JOIN gerber_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gerber' AND priority = 289;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    289 as priority
FROM openalex.awards.gerber_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.gerber_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_year
FROM openalex.awards.gerber_awards LIMIT 10;

In [ ]:
%sql
-- should all be Gerber Foundation
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.gerber_awards GROUP BY funder.display_name ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount)/1000000, 2) as funding_million_usd
FROM openalex.awards.gerber_awards GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    COUNT(lead_investigator) as has_pi,
    COUNT(start_year) as has_year,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount)/1000000, 2) as total_funding_million_usd
FROM openalex.awards.gerber_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as cnt
FROM openalex.awards.gerber_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
SELECT COUNT(*) as gerber_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gerber' AND priority = 289;